In [10]:
!pip uninstall -y xgboost
!pip install xgboost==2.0.3


Found existing installation: xgboost 2.0.3
Uninstalling xgboost-2.0.3:
  Successfully uninstalled xgboost-2.0.3
  Using cached xgboost-2.0.3-py3-none-manylinux2014_x86_64.whl.metadata (2.0 kB)
Using cached xgboost-2.0.3-py3-none-manylinux2014_x86_64.whl (297.1 MB)


In [1]:
!pip install scikit-learn==1.3.2

In [30]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.9/400.9 kB 8.0 MB/s eta 0:00:00


In [31]:
# 라이브러리 import

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold
from xgboost import XGBRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import joblib, json, shap, optuna
import scipy.sparse as sp
import seaborn as sns


In [32]:
# 버전 확인
print("pandas version: {}".format(pd.__version__))
print("NumPy version: {}".format(np.__version__))
print("scikit-learn version: {}".format(__import__('sklearn').__version__))
print("matplotlib version: {}".format(plt.matplotlib.__version__))
print("XGBoost version: {}".format(XGBRegressor.__module__.split('.')[0] + ' ' + __import__('xgboost').__version__))

pandas version: 2.2.2
NumPy version: 1.26.4
scikit-learn version: 1.3.2
matplotlib version: 3.10.0
XGBoost version: xgboost 2.0.3


In [33]:
# 데이터 로드
daegu_info = pd.read_csv('/content/drive/MyDrive/대구빅데이터분석/대구통합정보.csv')
fire_time = pd.read_csv('/content/drive/MyDrive/대구빅데이터분석/화재/화재_시간.csv')
fire_day  = pd.read_csv('/content/drive/MyDrive/대구빅데이터분석/화재/화재_요일.csv')
fire_month= pd.read_csv('/content/drive/MyDrive/대구빅데이터분석/화재/화재_월.csv')

In [34]:
# 컬럼명 한글 → 영문 매핌

# 1) 대구통합정보: 한글→영문 컬럼 매핑
daegu_info_colmap = {
    "시군구": "sigungu",
    "읍면동": "dong",
    "인구(명)": "population",
    "면적(㎢)": "area",
    "세대수" : "household",
    "인구밀도(인/㎢)": "pop_density",
    "평균 세대당 인수": "household_size",
    "자가주차장확보율" : "parking",
    "전통시장 수": "market",
    "일일차량 통행량": "traffic",
    "일일차량 평균속도": "speed",
    "4m미만_도로폭(km)" : "road_len_4",
    "4~5.5m_도로폭(km)": "road_len_5.5",
    "5.5~8m_도로폭(km)": "road_len_8",
    "8~12m_도로폭(km)": "road_len_12",
    "12m이상_도로폭(km)": "road_len_12up",
    "전체도로길이(km)": "road_len",
    "4m미만_도로폭비율": "ratio_road_len_4",
    "연평균구급출동거리" : "ambulance_len",
    "연평균구급출동횟수" :"ambulance_count",
    "연평균구조출동거리" : "rescue_len",
    "연평균구조출동횟수" :"rescue_count",
    "연평균생활안전출동거리" : "safety_len",
    "연평균생활안전출동횟수" :"safety_count",
    "연평균화재출동거리" : "fire_len",
    "연평균화재출동횟수" :"fire_count"
}

# 2) 화재(시간/요일/월)데이터: 한글→영문 컬럼 매핑
fire_hour_colmap = {
    "시군구": "sigungu",
    "읍면동": "dong",
    "시간": "hour",
    "시간별출동횟수": "count_hour",
    "시간별도착시간": "time_hour"
}

fire_day_colmap = {
    "시군구": "sigungu",
    "읍면동": "dong",
    "요일": "day",
    "요일별출동횟수": "count_day",
    "요일별도착시간": "time_day",
}

fire_month_colmap = {
    "시군구": "sigungu",
    "읍면동": "dong",
    "월": "month",
    "월별출동횟수": "count_month",
    "월별도착시간": "time_month"
}

# 3) 컬럼명 매핑 함수
def rename_cols(df, colmap):
    return df.rename(columns=colmap)

# 4) 적용
daegu_info  = rename_cols(daegu_info,  daegu_info_colmap)
fire_time   = rename_cols(fire_time,   fire_hour_colmap)
fire_day    = rename_cols(fire_day,    fire_day_colmap)
fire_month  = rename_cols(fire_month,  fire_month_colmap)


In [35]:
# 요일 한글 → 숫자 인코딩
day_map = {
    "월요일": 0,
    "화요일": 1,
    "수요일": 2,
    "목요일": 3,
    "금요일": 4,
    "토요일": 5,
    "일요일": 6
}

# 적용 (fire_day만 해당)
if "day" in fire_day.columns:
    fire_day["day"] = fire_day["day"].map(day_map)

In [36]:
# 데이터 병합

def merge_with_base_first(base, other, on='dong', how='left'):
    # 1) 오른쪽 DF에서 겹치는 컬럼 제거
    dup = set(base.columns).intersection(other.columns) - {on}
    other_wo = other.drop(columns=list(dup))

    # 2) 병합
    merged = base.merge(other_wo, on=on, how=how)
    base_cols = list(base.columns)
    extra_cols = [c for c in merged.columns if c not in base_cols]
    return merged[base_cols + extra_cols]

# 적용
fire_time_merged  = merge_with_base_first(daegu_info, fire_time,  on='dong')
fire_day_merged   = merge_with_base_first(daegu_info, fire_day,   on='dong')
fire_month_merged = merge_with_base_first(daegu_info, fire_month, on='dong')

In [28]:
# 모델 학습 및 평가

# --------------------------------------------
# CV + log1p 학습 함수
# --------------------------------------------
def run_cv_regression(df, target, features, preprocess, n_splits=5, random_state=42):
    """
    주어진 타깃(target)에 대해 교차검증(KFold) 기반으로
    log1p 변환 학습 → expm1 역변환 예측을 수행하고
    RMSE와 모델, 예측 결과를 반환
    """
    # 타깃이 있는 행만 학습에 사용 (결측 제거)
    mask_train = df[target].notna()
    df_train   = df.loc[mask_train].reset_index(drop=True)

    # 전체 피처(X), 학습용 피처(X_train), 타깃(y)
    X_full       = df[features]
    X_train      = df_train[features]
    y_train_raw  = df_train[target].values
    y_train_log1 = np.log1p(y_train_raw)  # 안정화를 위해 log1p 변환

    # K-Fold 교차검증 준비
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    oof_pred   = np.full(df_train.shape[0], np.nan, dtype=float)  # OOF 예측 결과 저장용
    models     = []   # fold별 학습된 모델
    fold_rmse  = []   # fold별 RMSE

    # ----- KFold 반복 학습 -----
    for fold, (tr_idx, va_idx) in enumerate(kf.split(X_train), 1):
        model = XGBRegressor(random_state=42)  # XGBoost 회귀 모델
        pipe  = Pipeline(steps=[('prep', preprocess), ('model', model)])  # 전처리 + 모델

        # fold별 train/valid 분할
        X_tr, X_va = X_train.iloc[tr_idx], X_train.iloc[va_idx]
        y_tr, y_va = y_train_log1[tr_idx], y_train_log1[va_idx]

        # 학습
        pipe.fit(X_tr, y_tr)

        # 검증 데이터 예측 (log1p 역변환 후 음수값은 0으로 클리핑)
        pred_va = pipe.predict(X_va)
        pred_va = np.expm1(pred_va)
        pred_va = np.clip(pred_va, a_min=0, a_max=None)

        # RMSE 계산
        rmse = mean_squared_error(np.expm1(y_va), pred_va, squared=False)
        fold_rmse.append(rmse)
        oof_pred[va_idx] = pred_va
        models.append(pipe)
        print(f'[{target}] Fold {fold}: RMSE {rmse:.4f}')

    # 전체 CV 결과 출력
    print(f'[{target}] CV RMSE (mean±std): {np.mean(fold_rmse):.4f} ± {np.std(fold_rmse):.4f}')

    # ----- 최종 모델 (전체 학습 데이터로 재학습) -----
    final_model = Pipeline(steps=[('prep', preprocess), ('model', XGBRegressor(random_state=42))])
    final_model.fit(X_train, y_train_log1)

    # 전체 데이터에 대한 예측 (추후 test 예측처럼 사용 가능)
    pred_all = np.expm1(final_model.predict(X_full))
    pred_all = np.clip(pred_all, a_min=0, a_max=None)  # 음수값 0으로 보정
    s_pred_all = pd.Series(pred_all, name=f'pred_{target}')

    # OOF 예측값 저장 (학습 데이터에만 있음)
    s_oof = pd.Series(np.nan, index=df.index, name=f'oof_{target}')
    s_oof.loc[df.index[mask_train]] = oof_pred

    # 결과 반환
    return {
        'models': models,           # fold별 모델
        'final_model': final_model, # 전체 학습 최종 모델
        'pred_all': s_pred_all,     # 전체 데이터 예측값
        'oof': s_oof,               # OOF 예측값
        'cv_rmse_mean': float(np.mean(fold_rmse)), # 평균 RMSE
        'cv_rmse_std' : float(np.std(fold_rmse))   # RMSE 표준편차
    }


# --------------------------------------------
# 동별 피처 SHAP 중요도 (상위 몇 개 동만)
# --------------------------------------------
def per_dong_feature_importance_shap(df, model_pipe, feature_cols, cat_onehot,
                                     top_dongs=10, top_features=10, save_csv=None):
    """
    동별로 SHAP 기반 피처 중요도를 계산하고
    상위 몇 개 동만 선택하여 주요 피처를 출력/저장
    """
    prep = model_pipe.named_steps['prep']
    xgb  = model_pipe.named_steps['model']

    # 최종 피처 이름 (원핫 변환된 것 + 그대로 통과된 것)
    onehot = prep.named_transformers_['cat']
    onehot_features = list(onehot.get_feature_names_out(cat_onehot))
    passthrough_features = [c for c in feature_cols if c not in cat_onehot]
    final_features = onehot_features + passthrough_features

    # 원본 피처명 → 최종 인덱스 매핑
    orig_to_idx = {}
    start = 0
    for cname in cat_onehot:
        cats = onehot.categories_[cat_onehot.index(cname)]
        count = len(cats)
        orig_to_idx[cname] = list(range(start, start+count))
        start += count
    for i, c in enumerate(passthrough_features, start=start):
        orig_to_idx[c] = [i]

    # 데이터 변환
    X_trans = prep.transform(df[feature_cols])
    if sp.issparse(X_trans):
        X_trans = X_trans.toarray()

    # SHAP 값 계산
    explainer = shap.TreeExplainer(xgb)
    shap_values = explainer.shap_values(X_trans)

    # 동별 전체 SHAP 크기 평균 → 중요 동 상위 10개 선정
    dong_scores = (pd.DataFrame({'dong': df['dong'].values, 'abs_shap_sum': np.abs(shap_values).sum(axis=1)})
                   .groupby('dong', as_index=False)['abs_shap_sum'].mean()
                   .sort_values('abs_shap_sum', ascending=False))
    top_dong_list = dong_scores['dong'].head(top_dongs).tolist()

    # ----- 동별 상세 피처 중요도 -----
    results = {}
    for dong_name in top_dong_list:
        mask = (df['dong'] == dong_name)
        shap_sub = shap_values[mask.values]

        rows = []
        for orig, idx_list in orig_to_idx.items():
            vals = np.abs(shap_sub[:, idx_list])
            contrib = vals.sum(axis=1) if vals.ndim == 2 else vals
            rows.append((orig, contrib.mean()))

        table = pd.DataFrame(rows, columns=['feature','mean_abs_shap']).sort_values('mean_abs_shap', ascending=False)
        results[dong_name] = table

        # 콘솔 출력
        print(f"\n[{dong_name}] 상위 {top_features} 피처")
        print(table.head(top_features))

    # 결과 CSV 저장 (동 10개 × 피처 10개)
    if save_csv:
        all_tables = []
        for dong, tbl in results.items():
            tbl2 = tbl.copy()
            tbl2.insert(0, 'dong', dong)
            all_tables.append(tbl2)
        pd.concat(all_tables, axis=0).to_csv(save_csv, index=False, encoding='utf-8-sig')
        print("Saved:", save_csv)

    return results


# --------------------------------------------
# 데이터셋 처리 함수 (학습 + 저장 + SHAP)
# --------------------------------------------
def train_on_dataset(df, df_name, labels, save_dir='/content/drive/MyDrive/대구빅데이터분석/화재'):
    """
    주어진 데이터셋(df)에 대해
    - LabelEncoding + OneHot 인코딩
    - 두 개 라벨 각각 학습
    - 모델/예측값/SHAP 결과 저장
    """
    print(f'\n===== Dataset: {df_name} =====')
    id_cols = []
    feature_cols = [c for c in df.columns if c not in labels + id_cols]

    # dong → LabelEncoding (150개 카테고리 → 숫자)
    le_dong = LabelEncoder()
    df['dong_le'] = le_dong.fit_transform(df['dong'])
    feature_cols = [c if c != 'dong' else 'dong_le' for c in feature_cols]

    # 문자형 피처만 원핫 (dong은 제외)
    obj_cols = [c for c in feature_cols if df[c].dtype == 'object']
    cat_onehot = [c for c in obj_cols if c != 'dong']

    # 전처리: cat_onehot → 원핫, 나머지 → 그대로 통과
    preprocess = ColumnTransformer(
        transformers=[('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=True), cat_onehot)],
        remainder='passthrough'
    )

    # 두 개 라벨 각각 학습 (예: count_hour, time_hour)
    res_a = run_cv_regression(df, labels[0], feature_cols, preprocess)
    res_b = run_cv_regression(df, labels[1], feature_cols, preprocess)

    # 예측 결과 합치기 (원본 df + 예측값 + OOF)
    df_out = pd.concat([df, res_a['pred_all'], res_b['pred_all'], res_a['oof'], res_b['oof']], axis=1)

    # 모델/예측 결과 저장
    path_model_a = f'{save_dir}/model_{df_name}_{labels[0]}.joblib'
    path_model_b = f'{save_dir}/model_{df_name}_{labels[1]}.joblib'
    joblib.dump(res_a['final_model'], path_model_a)
    joblib.dump(res_b['final_model'], path_model_b)
    path_pred = f'{save_dir}/{df_name}_pred.csv'
    df_out.to_csv(path_pred, index=False, encoding='utf-8-sig')
    print('Saved:', path_model_a, path_model_b, path_pred)

    # 동별 SHAP 상위 10개 동 × 피처 10개 출력/저장
    save_csv = f'{save_dir}/{df_name}_dong_top10_features.csv'
    per_dong_feature_importance_shap(df, res_a['final_model'], feature_cols, cat_onehot,
                                     top_dongs=10, top_features=10, save_csv=save_csv)

    return {
        'res_a': res_a, 'res_b': res_b,
        'feature_cols': feature_cols,
        'le_dong': le_dong
    }


# =========================
# 실행 (세 개 데이터셋 각각)
# =========================
time_labels  = ['count_hour',  'time_hour']
day_labels   = ['count_day',   'time_day']
month_labels = ['count_month', 'time_month']

out_time  = train_on_dataset(fire_time_merged.copy(),  'fire_time',  time_labels)
out_day   = train_on_dataset(fire_day_merged.copy(),   'fire_day',   day_labels)
out_month = train_on_dataset(fire_month_merged.copy(), 'fire_month', month_labels)



===== Dataset: fire_time =====
[count_hour] Fold 1: RMSE 0.5355
[count_hour] Fold 2: RMSE 0.6001
[count_hour] Fold 3: RMSE 0.6343
[count_hour] Fold 4: RMSE 0.5949
[count_hour] Fold 5: RMSE 0.5990
[count_hour] CV RMSE (mean±std): 0.5927 ± 0.0320
[time_hour] Fold 1: RMSE 1.6041
[time_hour] Fold 2: RMSE 1.6166
[time_hour] Fold 3: RMSE 1.7771
[time_hour] Fold 4: RMSE 2.6700
[time_hour] Fold 5: RMSE 1.4769
[time_hour] CV RMSE (mean±std): 1.8290 ± 0.4312
Saved: /content/drive/MyDrive/대구빅데이터분석/화재/model_fire_time_count_hour.joblib /content/drive/MyDrive/대구빅데이터분석/화재/model_fire_time_time_hour.joblib /content/drive/MyDrive/대구빅데이터분석/화재/fire_time_pred.csv

[대명9동] 상위 10 피처
             feature  mean_abs_shap
0            sigungu       0.246553
26              hour       0.190863
25        fire_count       0.102747
19   ambulance_count       0.078517
16          road_len       0.072400
10             speed       0.044802
17  ratio_road_len_4       0.035242
15     road_len_12up       0.031823
21     

In [39]:
# 하이퍼파라미터 튜닝 및 최종 모델 학습 후 저장

# ============================================
# 교차검증(CV) + log1p 변환 학습 함수
# ============================================
def run_cv_regression(df, target, features, preprocess, best_params,
                      n_splits=5, random_state=42):
    """
    주어진 타깃(target)에 대해:
      1) log1p 변환하여 안정적으로 학습
      2) 교차검증(KFold)으로 RMSE 평가
      3) 최종 모델은 전체 데이터로 재학습
      4) 예측값(pred_all), OOF 예측(oof), RMSE 반환
    """
    # -------------------------
    # 데이터 준비
    # -------------------------
    mask_train = df[target].notna()                    # 타깃값이 있는 행만 학습에 사용
    df_train   = df.loc[mask_train].reset_index(drop=True)

    X_full       = df[features]                        # 전체 feature (예측용)
    X_train      = df_train[features]                  # 학습 feature
    y_train_raw  = df_train[target].values
    y_train_log1 = np.log1p(y_train_raw)               # 타깃값 log1p 변환

    # -------------------------
    # KFold 교차검증
    # -------------------------
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    oof_pred   = np.full(df_train.shape[0], np.nan, dtype=float)  # fold별 OOF 예측 저장
    models     = []   # fold별 학습된 모델
    fold_rmse  = []   # fold별 RMSE 기록

    for fold, (tr_idx, va_idx) in enumerate(kf.split(X_train), 1):
        # XGBoost 모델 생성 (Optuna 튜닝된 best_params 적용)
        model = XGBRegressor(**best_params, random_state=42, tree_method="hist")
        pipe  = Pipeline(steps=[('prep', preprocess), ('model', model)])  # 전처리 + 모델

        # 학습/검증 데이터 분리
        X_tr, X_va = X_train.iloc[tr_idx], X_train.iloc[va_idx]
        y_tr, y_va = y_train_log1[tr_idx], y_train_log1[va_idx]

        # 모델 학습
        pipe.fit(X_tr, y_tr)

        # 검증 데이터 예측 (log1p 역변환 후 음수 → 0 보정)
        pred_va = np.expm1(pipe.predict(X_va))
        pred_va = np.clip(pred_va, a_min=0, a_max=None)

        # RMSE 계산
        rmse = mean_squared_error(np.expm1(y_va), pred_va, squared=False)
        fold_rmse.append(rmse)
        oof_pred[va_idx] = pred_va
        models.append(pipe)
        print(f'[{target}] Fold {fold}: RMSE {rmse:.4f}')

    # 교차검증 결과 요약
    print(f'[{target}] CV RMSE (mean±std): {np.mean(fold_rmse):.4f} ± {np.std(fold_rmse):.4f}')

    # -------------------------
    # 최종 모델: 전체 데이터로 학습
    # -------------------------
    final_model = Pipeline(steps=[('prep', preprocess),
                                  ('model', XGBRegressor(**best_params, random_state=42, tree_method="hist"))])
    final_model.fit(X_train, y_train_log1)

    # 전체 데이터 예측
    pred_all = np.expm1(final_model.predict(X_full))
    pred_all = np.clip(pred_all, a_min=0, a_max=None)
    s_pred_all = pd.Series(pred_all, name=f'pred_{target}')

    # OOF 예측값 저장
    s_oof = pd.Series(np.nan, index=df.index, name=f'oof_{target}')
    s_oof.loc[df.index[mask_train]] = oof_pred

    return {
        'models': models,           # fold별 학습 모델
        'final_model': final_model, # 전체 학습 최종 모델
        'pred_all': s_pred_all,     # 전체 예측값
        'oof': s_oof,               # OOF 예측값
        'cv_rmse_mean': float(np.mean(fold_rmse)), # 평균 RMSE
        'cv_rmse_std' : float(np.std(fold_rmse))   # 표준편차
    }


# ============================================
# Optuna 튜닝 (최적 하이퍼파라미터 탐색)
# ============================================
def objective(trial, X, y, preprocess, n_splits=5):
    """
    Optuna에서 trial별로 제안된 하이퍼파라미터를 적용하여
    교차검증 RMSE를 반환 (최소화 목표)
    """
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 800),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "tree_method": "hist",
        "random_state": 42
    }

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    rmses = []

    for tr_idx, va_idx in kf.split(X):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        pipe = Pipeline(steps=[('prep', preprocess),
                               ('model', XGBRegressor(**params))])
        pipe.fit(X_tr, y_tr)

        pred_va = np.expm1(pipe.predict(X_va))
        pred_va = np.clip(pred_va, a_min=0, a_max=None)
        rmse = mean_squared_error(np.expm1(y_va), pred_va, squared=False)
        rmses.append(rmse)

    return np.mean(rmses)


def tune_params(df, target, features, preprocess, n_trials=30):
    """
    주어진 target에 대해 Optuna로 n_trials번 탐색하여
    최적의 하이퍼파라미터 반환
    """
    mask_train = df[target].notna()
    df_train   = df.loc[mask_train].reset_index(drop=True)
    X = df_train[features]
    y = np.log1p(df_train[target].values)

    study = optuna.create_study(direction="minimize")   # RMSE 최소화 목표
    study.optimize(lambda trial: objective(trial, X, y, preprocess), n_trials=n_trials)

    print("Best Trial:", study.best_trial.number)
    print("Best Params:", study.best_trial.params)
    print("Best RMSE:", study.best_value)

    return study.best_trial.params


# ============================================
# 동별 SHAP 중요도 분석
# ============================================
def per_dong_feature_importance_shap(df, model_pipe, feature_cols, cat_onehot,
                                     top_dongs=10, top_features=10, save_csv=None):
    """
    각 동별로 SHAP값을 계산해서,
    영향력이 큰 상위 10개 동에 대해 피처 중요도를 출력/저장
    """
    prep = model_pipe.named_steps['prep']
    xgb  = model_pipe.named_steps['model']

    onehot = prep.named_transformers_['cat']
    onehot_features = list(onehot.get_feature_names_out(cat_onehot))
    passthrough_features = [c for c in feature_cols if c not in cat_onehot]

    # 원본 피처명 → 최종 인덱스 매핑
    orig_to_idx = {}
    start = 0
    for cname in cat_onehot:
        cats = onehot.categories_[cat_onehot.index(cname)]
        count = len(cats)
        orig_to_idx[cname] = list(range(start, start+count))
        start += count
    for i, c in enumerate(passthrough_features, start=start):
        orig_to_idx[c] = [i]

    # 전처리 후 데이터 변환
    X_trans = prep.transform(df[feature_cols])
    if sp.issparse(X_trans):
        X_trans = X_trans.toarray()

    # SHAP 계산
    explainer = shap.TreeExplainer(xgb)
    shap_values = explainer.shap_values(X_trans)

    # 동별 평균 SHAP 합 → 중요 동 상위 10개
    dong_scores = (pd.DataFrame({'dong': df['dong'].values,
                                 'abs_shap_sum': np.abs(shap_values).sum(axis=1)})
                   .groupby('dong', as_index=False)['abs_shap_sum'].mean()
                   .sort_values('abs_shap_sum', ascending=False))

    top_dong_list = dong_scores['dong'].head(top_dongs).tolist()
    results = {}

    # 각 동별 주요 피처 출력
    for dong_name in top_dong_list:
        mask = (df['dong'] == dong_name)
        shap_sub = shap_values[mask.values]

        rows = []
        for orig, idx_list in orig_to_idx.items():
            vals = np.abs(shap_sub[:, idx_list])
            contrib = vals.sum(axis=1) if vals.ndim == 2 else vals
            rows.append((orig, contrib.mean()))

        table = pd.DataFrame(rows, columns=['feature','mean_abs_shap']).sort_values('mean_abs_shap', ascending=False)
        results[dong_name] = table
        print(f"\n[{dong_name}] 상위 {top_features} 피처")
        print(table.head(top_features))

    # CSV 저장
    if save_csv:
        all_tables = []
        for dong, tbl in results.items():
            tbl2 = tbl.copy()
            tbl2.insert(0, 'dong', dong)
            all_tables.append(tbl2)
        pd.concat(all_tables, axis=0).to_csv(save_csv, index=False, encoding='utf-8-sig')
        print("Saved:", save_csv)

    return results


# ============================================
# 데이터셋 처리 (튜닝 → 학습 → 저장 → SHAP)
# ============================================
def train_on_dataset(df, df_name, labels, save_dir='/content/drive/MyDrive/대구빅데이터분석/화재'):
    """
    1) LabelEncoding & OneHotEncoding 전처리
    2) Optuna로 target별 하이퍼파라미터 튜닝
    3) 최종 모델 학습 & 저장
    4) SHAP 중요도 분석
    """
    print(f'\n===== Dataset: {df_name} =====')
    feature_cols = [c for c in df.columns if c not in labels]

    # dong을 숫자로 변환 (LabelEncoder)
    le_dong = LabelEncoder()
    df['dong_le'] = le_dong.fit_transform(df['dong'])
    feature_cols = [c if c != 'dong' else 'dong_le' for c in feature_cols]

    # 문자형 피처만 OneHot 인코딩
    obj_cols = [c for c in feature_cols if df[c].dtype == 'object']
    cat_onehot = [c for c in obj_cols if c != 'dong']

    preprocess = ColumnTransformer(
        transformers=[('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=True), cat_onehot)],
        remainder='passthrough'
    )

    results = {}
    for target in labels:
        # (1) Optuna 하이퍼파라미터 튜닝
        best_params = tune_params(df, target, feature_cols, preprocess, n_trials=30)
        # (2) 최적 파라미터로 교차검증 학습
        res = run_cv_regression(df, target, feature_cols, preprocess, best_params)
        results[target] = res

    # 예측 결과 합치기
    df_out = pd.concat([df] + [results[t]['pred_all'] for t in labels] +
                       [results[t]['oof'] for t in labels], axis=1)

    # 모델 저장
    for target in labels:
        path_model = f'{save_dir}/model_{df_name}_{target}.joblib'
        joblib.dump(results[target]['final_model'], path_model)
    path_pred = f'{save_dir}/{df_name}_pred.csv'
    df_out.to_csv(path_pred, index=False, encoding='utf-8-sig')
    print('Saved:', path_pred)

    # SHAP 저장
    save_csv = f'{save_dir}/{df_name}_dong_top10_features.csv'
    per_dong_feature_importance_shap(df, results[labels[0]]['final_model'], feature_cols, cat_onehot,
                                     top_dongs=10, top_features=10, save_csv=save_csv)

    return results


# ============================================
# 실행 (세 개 데이터셋 각각)
# ============================================
time_labels  = ['count_hour',  'time_hour']
day_labels   = ['count_day',   'time_day']
month_labels = ['count_month', 'time_month']

out_time  = train_on_dataset(fire_time_merged.copy(),  'fire_time',  time_labels)
out_day   = train_on_dataset(fire_day_merged.copy(),   'fire_day',   day_labels)
out_month = train_on_dataset(fire_month_merged.copy(), 'fire_month', month_labels)


[I 2025-10-08 17:20:35,439] A new study created in memory with name: no-name-e5599b7a-0151-4379-b933-c4e85702102e



===== Dataset: fire_time =====


[I 2025-10-08 17:20:49,677] Trial 0 finished with value: 0.5636243363821061 and parameters: {'n_estimators': 648, 'max_depth': 6, 'learning_rate': 0.045993472604242684, 'subsample': 0.8069216063090122, 'colsample_bytree': 0.785928745699602, 'min_child_weight': 9, 'reg_alpha': 6.977235121224361e-06, 'reg_lambda': 2.6899903630168604e-05}. Best is trial 0 with value: 0.5636243363821061.
[I 2025-10-08 17:21:08,153] Trial 1 finished with value: 0.5801136923431595 and parameters: {'n_estimators': 463, 'max_depth': 9, 'learning_rate': 0.0400871166884648, 'subsample': 0.7117665224176788, 'colsample_bytree': 0.8358809243300522, 'min_child_weight': 6, 'reg_alpha': 4.210628649421763e-06, 'reg_lambda': 4.192474022950623e-08}. Best is trial 0 with value: 0.5636243363821061.
[I 2025-10-08 17:21:48,262] Trial 2 finished with value: 0.6229863364877481 and parameters: {'n_estimators': 792, 'max_depth': 9, 'learning_rate': 0.2324119723547855, 'subsample': 0.8779672801452085, 'colsample_bytree': 0.847087

Best Trial: 27
Best Params: {'n_estimators': 205, 'max_depth': 5, 'learning_rate': 0.01680193598278528, 'subsample': 0.8288627332643901, 'colsample_bytree': 0.9963626979870089, 'min_child_weight': 6, 'reg_alpha': 1.320910232538362e-06, 'reg_lambda': 1.6048429241055e-07}
Best RMSE: 0.5373900617240632
[count_hour] Fold 1: RMSE 0.4794
[count_hour] Fold 2: RMSE 0.5624
[count_hour] Fold 3: RMSE 0.5577
[count_hour] Fold 4: RMSE 0.5558
[count_hour] Fold 5: RMSE 0.5316
[count_hour] CV RMSE (mean±std): 0.5374 ± 0.0309


[I 2025-10-08 17:25:24,616] A new study created in memory with name: no-name-84a564ac-6722-43aa-9b43-bddd0b01cd41
[I 2025-10-08 17:25:33,417] Trial 0 finished with value: 1.9928350668986041 and parameters: {'n_estimators': 390, 'max_depth': 9, 'learning_rate': 0.2054459704408109, 'subsample': 0.7058225580197505, 'colsample_bytree': 0.8640915078049163, 'min_child_weight': 5, 'reg_alpha': 1.3112267350275637e-08, 'reg_lambda': 7.30059646033823e-07}. Best is trial 0 with value: 1.9928350668986041.
[I 2025-10-08 17:25:35,371] Trial 1 finished with value: 1.9499079119537164 and parameters: {'n_estimators': 478, 'max_depth': 3, 'learning_rate': 0.06859679084498307, 'subsample': 0.6213546724447575, 'colsample_bytree': 0.6762721237275735, 'min_child_weight': 6, 'reg_alpha': 4.9865420339577314e-08, 'reg_lambda': 0.04166043144004065}. Best is trial 1 with value: 1.9499079119537164.
[I 2025-10-08 17:25:56,030] Trial 2 finished with value: 1.8676765747015267 and parameters: {'n_estimators': 489, 'm

...


[다사읍] 상위 10 피처
          feature  mean_abs_shap
25     fire_count       0.407378
15  road_len_12up       0.203127
4       household       0.186110
14    road_len_12       0.127893
16       road_len       0.107260
23   safety_count       0.092210
13     road_len_8       0.082500
26            day       0.057028
18  ambulance_len       0.037332
24       fire_len       0.036020

[신당동] 상위 10 피처
            feature  mean_abs_shap
25       fire_count       0.371946
4         household       0.225605
15    road_len_12up       0.195139
14      road_len_12       0.114007
16         road_len       0.112456
19  ambulance_count       0.099674
6    household_size       0.073304
26              day       0.034848
12     road_len_5.5       0.033732
18    ambulance_len       0.024730

[구지면] 상위 10 피처
            feature  mean_abs_shap
25       fire_count       0.350317
15    road_len_12up       0.153269
23     safety_count       0.103529
14      road_len_12       0.094624
16         road_len       0.0

[I 2025-10-08 17:37:27,077] Trial 0 finished with value: 0.8519529475644042 and parameters: {'n_estimators': 642, 'max_depth': 8, 'learning_rate': 0.02815061025289048, 'subsample': 0.7305966169949009, 'colsample_bytree': 0.7535115060827752, 'min_child_weight': 10, 'reg_alpha': 1.6285808120108244e-07, 'reg_lambda': 3.825257814480665e-06}. Best is trial 0 with value: 0.8519529475644042.
[I 2025-10-08 17:37:35,001] Trial 1 finished with value: 0.8312200055006423 and parameters: {'n_estimators': 768, 'max_depth': 6, 'learning_rate': 0.017051475658306826, 'subsample': 0.8351179149006459, 'colsample_bytree': 0.6675963467370292, 'min_child_weight': 6, 'reg_alpha': 0.00010094246715556915, 'reg_lambda': 1.3757055224747968e-07}. Best is trial 1 with value: 0.8312200055006423.
[I 2025-10-08 17:37:39,232] Trial 2 finished with value: 0.8105689760707527 and parameters: {'n_estimators': 232, 'max_depth': 3, 'learning_rate': 0.04369643004804419, 'subsample': 0.9703348132650966, 'colsample_bytree': 0.

Best Trial: 23
Best Params: {'n_estimators': 293, 'max_depth': 4, 'learning_rate': 0.016502624350366635, 'subsample': 0.942679406621179, 'colsample_bytree': 0.8751663279614093, 'min_child_weight': 5, 'reg_alpha': 0.00045641284288670315, 'reg_lambda': 7.6249538312093075e-06}
Best RMSE: 0.8081670731818731
[count_month] Fold 1: RMSE 0.7408
[count_month] Fold 2: RMSE 0.9011
[count_month] Fold 3: RMSE 0.7894
[count_month] Fold 4: RMSE 0.8489
[count_month] Fold 5: RMSE 0.7607
[count_month] CV RMSE (mean±std): 0.8082 ± 0.0591


[I 2025-10-08 17:41:00,027] A new study created in memory with name: no-name-5fd5d4bc-ea46-4bc4-a83f-e039f3815eae
[I 2025-10-08 17:41:07,903] Trial 0 finished with value: 2.170711354849158 and parameters: {'n_estimators': 576, 'max_depth': 7, 'learning_rate': 0.04247393436865924, 'subsample': 0.9866249351178145, 'colsample_bytree': 0.6102365122976966, 'min_child_weight': 3, 'reg_alpha': 1.8065256571152044e-05, 'reg_lambda': 1.8176720415360172}. Best is trial 0 with value: 2.170711354849158.
[I 2025-10-08 17:41:13,247] Trial 1 finished with value: 2.5154585813671373 and parameters: {'n_estimators': 539, 'max_depth': 3, 'learning_rate': 0.23326516112458073, 'subsample': 0.9115254598707763, 'colsample_bytree': 0.7323467371493622, 'min_child_weight': 10, 'reg_alpha': 1.4119382011514213e-05, 'reg_lambda': 0.00010238707402731086}. Best is trial 0 with value: 2.170711354849158.
[I 2025-10-08 17:41:15,258] Trial 2 finished with value: 2.254808001163865 and parameters: {'n_estimators': 464, 'ma

Best Trial: 11
Best Params: {'n_estimators': 778, 'max_depth': 10, 'learning_rate': 0.010791593663715331, 'subsample': 0.9721855948502076, 'colsample_bytree': 0.8493769475524452, 'min_child_weight': 1, 'reg_alpha': 0.0006918103595293587, 'reg_lambda': 5.060272209237385}
Best RMSE: 2.1359647060318983
[time_month] Fold 1: RMSE 2.0537
[time_month] Fold 2: RMSE 1.9906
[time_month] Fold 3: RMSE 1.9219
[time_month] Fold 4: RMSE 2.2617
[time_month] Fold 5: RMSE 2.4519
[time_month] CV RMSE (mean±std): 2.1360 ± 0.1946
Saved: /content/drive/MyDrive/대구빅데이터분석/화재/fire_month_pred.csv

[대명9동] 상위 10 피처
            feature  mean_abs_shap
0           sigungu       0.271277
25       fire_count       0.210014
19  ambulance_count       0.097189
16         road_len       0.077318
13       road_len_8       0.063307
10            speed       0.042792
7           parking       0.041568
4         household       0.038356
26            month       0.035917
18    ambulance_len       0.035187

[신당동] 상위 10 피처
     